In [ ]:
# If running on Colab, uncomment:
# !pip install tensorflow xgboost scikit-learn seaborn

import sys
sys.path.append("..")   # so we can import from model/

from model.lstm_xgboost import (
    load_data, build_tokenizer, encode,
    build_lstm, build_feature_extractor, extract_features,
    build_xgboost, evaluate,
    plot_confusion_matrix, plot_roc,
    print_comparison_table, FakeNewsDetector,
    SEED, TEST_SIZE
)
from sklearn.model_selection import train_test_split

In [ ]:
# Download from: https://www.kaggle.com/competitions/fake-news/data
# Place train.csv in ../data/

df = load_data("../data/train.csv")
print(f"Total samples : {len(df):,}")
print(f"Fake news     : {df['label'].sum():,}")
print(f"Real news     : {(df['label']==0).sum():,}")
df.head()

In [ ]:
X_text = df["content"].tolist()
y      = df["label"].values

X_tr, X_val, y_tr, y_val = train_test_split(
    X_text, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)
print(f"Train: {len(X_tr):,}  |  Val: {len(X_val):,}")

tokenizer  = build_tokenizer(X_tr)
vocab_size = min(50_000, len(tokenizer.word_index) + 1)
print(f"Vocab size: {vocab_size:,}")

X_tr_pad  = encode(tokenizer, X_tr)
X_val_pad = encode(tokenizer, X_val)
print(f"Input shape: {X_tr_pad.shape}")

In [ ]:
lstm_model = build_lstm(vocab_size)
lstm_model.summary()

history = lstm_model.fit(
    X_tr_pad, y_tr,
    validation_data=(X_val_pad, y_val),
    epochs=5,
    batch_size=64,
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["accuracy"],     label="Train")
axes[0].plot(history.history["val_accuracy"], label="Val")
axes[0].set_title("Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"],     label="Train")
axes[1].plot(history.history["val_loss"], label="Val")
axes[1].set_title("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

y_prob_lstm = lstm_model.predict(X_val_pad, batch_size=64).flatten()
y_pred_lstm = (y_prob_lstm >= 0.5).astype(int)

lstm_metrics = evaluate(y_val, y_pred_lstm, y_prob_lstm, "BiLSTM standalone")
plot_confusion_matrix(y_val, y_pred_lstm, title="BiLSTM — Confusion Matrix")

In [ ]:
extractor = build_feature_extractor(lstm_model)
feats_tr  = extract_features(extractor, X_tr_pad)
feats_val = extract_features(extractor, X_val_pad)

print(f"Feature shape: {feats_tr.shape}")

xgb_model = build_xgboost()
xgb_model.fit(feats_tr, y_tr, eval_set=[(feats_val, y_val)], verbose=False)
print("XGBoost training complete.")

In [ ]:
y_prob_hybrid = xgb_model.predict_proba(feats_val)[:, 1]
y_pred_hybrid = xgb_model.predict(feats_val)

hybrid_metrics = evaluate(y_val, y_pred_hybrid, y_prob_hybrid, "LSTM + XGBoost Hybrid")
plot_confusion_matrix(y_val, y_pred_hybrid, title="Hybrid — Confusion Matrix")
plot_roc(y_val, y_prob_hybrid)
print_comparison_table(hybrid_metrics)

In [ ]:
detector = FakeNewsDetector(lstm_model, xgb_model, tokenizer)

test_articles = [
    "Scientists at MIT have developed a new battery technology that could double EV range.",
    "SHOCKING: Government secretly putting microchips in vaccines, whistleblower reveals.",
    "Federal Reserve raises interest rates by 25 basis points amid inflation concerns.",
    "Celebrity seen shape-shifting at public event, video goes viral."
]

print(f"\n{'─'*60}")
print(f"  {'Article':<42} {'Label':<6} {'Conf':>6}")
print(f"{'─'*60}")
for article in test_articles:
    label, conf = detector.predict(article)
    short = article[:40] + "..."
    print(f"  {short:<43} {label:<6} {conf:.1%}")
print(f"{'─'*60}")